# Task 4.2 - Valutazione di Naive-Bayes su `training_clean.csv`

Nel Task 2.2 abbiamo applicato il classificatore di Naive-Bayes su `manuale.csv` prima manualmente e poi in python. Quella valutazione è però poco significativa, perchè fatta su un dataset di appena 14 campioni che non è pienamente rappresentativo dei dati contenuti in `training.csv`.

In questo notebook applichiamo Naive-Bayes ai 49.799 campioni di
`training_clean.csv` (il dataset ripulito ottenuto dal Task 3), che il modello non ha mai visto
e che hanno la proporzione reale delle classi (~99% classe 0, ~1% dropout).

Il lavoro si divide in cinque parti:
1. Applicare Naive-Bayes senza modificarlo e misurarne le prestazioni;
2. Spiegare i risultati ottenuti;
3. Valutare le prestazioni;
4. Provare a migliorare le prestazioni del classificatore;
4. Trarre le conclusioni in vista del Task 5.

## 1 - Applicazione del classificatore di Naive-Bayes sul dataset `training_clean.csv`
Riprendiamo il modello del Task 2.2 *senza modificarlo*: le probabilità restano
quelle stimate su `manuale.csv` (stimatore di Laplace e valori di *k* calcolati
sull'intero dataset). L'obiettivo non è ri-addestrare, ma misurare come si comporta su
dati nuovi il modello costruito sui 14 campioni.

Per efficienza, invece di ricontare le frequenze su `manuale.csv` a ogni riga (lento su
50.000 osservazioni), le calcoliamo **una volta sola** e le salviamo in una tabella di
probabilità: il risultato è identico al Task 2.2, solo più veloce.

### 1.1 - Carica dati e parametri

In [1]:
import pandas as pd

# dati su cui valutiamo: training_clean.csv (output del Task 3)
df = pd.read_csv("../data/training_clean.csv")

# sorgente del modello: le frequenze restano quelle di manuale.csv (Task 2.2)
man = pd.read_csv("../data/manuale.csv")

FEATS = ["FEATURE0", "FEATURE1", "FEATURE2", "FEATURE3"]
K = {"FEATURE0": 26, "FEATURE1": 16, "FEATURE2": 87, "FEATURE3": 164}
n1 = (man["LABEL"] == 1).sum()
n0 = (man["LABEL"] == 0).sum()
N = len(man)

print("training_clean:", df.shape)
print(df["LABEL"].value_counts().to_dict())

training_clean: (49799, 9)
{0: 49309, 1: 490}


### 1.2 - Tabelle di probabilità del modello, con Laplace

In [2]:
# prob[classe][feature][valore] = P(feature = valore | classe), stimata su manuale.csv
# E' la formula di Laplace del Task 2.2: (conteggio + 1) / (n_classe + k)
prob = {1: {}, 0: {}}
for f in FEATS:
    for classe, n in ((1, n1), (0, n0)):
        tabella = {}
        for v in man[f].unique():
            conteggio = ((man[f] == v) & (man["LABEL"] == classe)).sum()
            tabella[v] = (conteggio + 1) / (n + K[f])
        # valore mai visto in manuale.csv: conteggio = 0, quindi 1 / (n_classe + k)
        tabella["_mai_visto_"] = 1 / (n + K[f])
        prob[classe][f] = tabella

### 1.3 - Applica il classificatore a training_clean

In [3]:
def score(riga, classe):
    """Priori x prodotto delle P(feature | classe), lette dalle tabelle sopra."""
    risultato = (n1 if classe == 1 else n0) / N
    for f in FEATS:
        tabella = prob[classe][f]
        # se il valore non è tra quelli visti in manuale, uso il caso "_mai_visto_"
        risultato *= tabella.get(riga[f], tabella["_mai_visto_"])
    return risultato

def classifica(riga):
    return 1 if score(riga, 1) > score(riga, 0) else 0

df["PREDETTA"] = df.apply(classifica, axis=1)

# prima verifica: quante righe finiscono in ciascuna classe?
df["PREDETTA"].value_counts()

PREDETTA
1    48520
0     1279
Name: count, dtype: int64

## 2 - Considerazioni sui risultati ottenuti
Confrontiamo la distribuzione delle predizioni con quella reale, poi vediamo perché il modello si comporta così.

## 3 - Valutazione delle prestazioni

## 4 - Ottimizzazione delle prestazioni

## 5 - Conclusioni